# Initializing kcat values for PAMparametrizer

In this notebook, we give an overview of how an initial set of turnover number can be obtained. To enable this, we use multiple databases to map kcat values obtained from machine learning (from the [GotEnzymes](https://metabolicatlas.org/gotenzymes) database) to the proteins and reactions in the model using the gene-protein-reaction associations.

Before we start, we have to set up all the package and files required. Adjust the filenames and paths as desired. Importantly, make sure to **adjust the regular expression in the cell below** in order to find the gene names associated with the organism you want to model.

In [1]:
import pandas as pd
import os
from cobra.io import read_sbml_model
import matplotlib.pyplot as plt
import re
import numpy as np
import datetime

if os.path.split(os.getcwd())[1]=='i1_preprocessing':
    os.chdir('../..')
    
from PAModelpy.utils.pam_generation import merge_enzyme_complexes, get_protein_gene_mapping
from Modules.utils.preprocessing import *

PAM_DATA_FILE_PATH = os.path.join('Data', 'proteinAllocationModel_iML1515_EnzymaticData_new.xls')
UNIPROT_FILE_PATH = os.path.join('Data', 'Databases','uniprotkb_ecolik12_240726.xlsx')
GOTENZYMES_FILE_PATH = os.path.join('Data','Databases', 'GotEnzymes_eco_230309.json')

Loading PAModelpy modules version 0.0.5.1
Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-03


In [2]:
#output file path
# Create a datetime object for the current date
current_date = datetime.datetime.now()
# Format the date as yymmdd
formatted_date = current_date.strftime('%y%m%d')

OUTPUT_FILE_PATH = os.path.join('Results', '1_preprocessing', f'proteinAllocationModel_iML1515_EnzymaticData_{formatted_date}.xlsx')

In [3]:
#You need to adjust this to find the geneid/locustag for your microbe
locustag_regex =r'\b([b|s]\d{4})\b'

## 0. Download information from different databases
1. **Metabolic model stoichiometries and reactions: [BiGG database](http://bigg.ucsd.edu/)**
- Get your model! Or use your own reaction identifiers
- In this example we will use the iML1515 model for [*Escherichia coli* K12 substr. MG1655](http://bigg.ucsd.edu/models/iML1515)
- Important: the identifiers must me mappable to uniprot (or another enzyme identifier database) and KEGG (for mapping to GotEnzymes)

2. **Turnover numbers: [GotEnzymes](https://metabolicatlas.org/gotenzymes)**
- Go to the [API of the Metabolic Atlas](https://metabolicatlas.org/api/v2/#/GotEnzymes/gotEnzymes)
- Use the `GET\gotenzymes\enzymes?` functionality, click `Try it out`
- Here we filled in `eco` (*E. coli* strain K12) for `organism` and `10000` for `pagination[pageSize]`. The rest was left empty
- Click execute and a json file with all the information will be downloaded
- Note: GotEnzymes gives the locus tag, ec number, reaction id and compound for each protein. The latter 2 are given as KEGG identifiers, which we subsequently have to map to BiGG identifiers in order to match with the model. Therefore, we need to use other databases

3. **Gene-Protein-Reaction relations: [UniprotKB](https://www.uniprot.org/)**
- Got to the [Uniprot knowledge base](https://www.uniprot.org/) and used advanced search to query for your organism. For *E. coli* we used [this](https://www.uniprot.org/uniprotkb?query=%28organism_id%3A83333%29) webpage
- Click download, select `format`:`Excel` and select the following: 
    - `Uniprot Data -> Names & Taxonomy`: `Gene Names`
    - `Uniprot Data -> Sequences`: `length`, `mass`
    - `Uniprot Data -> Function`: `RheaID`
- Download the resulting Excel file

## 1. Get information from the BiGG model

In the metabolic model itself is already quite some information stored. In order to obtain up-to-date information on the enzymes related to the reaction, we will need to obtain the kegg reaction ID, the reactants, the product and the gene-protein relation.

In [4]:
#load the model
model = read_sbml_model(os.path.join('Models', 'iML1515.xml'))

#make a id mapping and create a mapping dataframe
id_mapper_df = create_id_mapper_from_model(model)
id_mapper_df.head()

Mapped 745 of 2053 reactions to KEGG reaction IDs.


,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,ec-code
0,CYTDK2,"[C00475, C00044]","[C00055, C00035, C00080]",b2066,False,R00517,"[2.7.1.48, 2.7.1.213]"
1,XPPT,"[C00119, C00385]","[C00013, C00655]",b0238,False,R02142,"[2.4.2.8, 2.4.2.22]"
2,HXPRT,"[C00262, C00119]","[C00130, C00013]",b0238 or b0125,False,R01132,2.4.2.8
3,NDPK5,"[C00002, C00361]","[C00008, C00286]",b0474 or b2518,True,R01857,2.7.4.6
4,SHK3Dr,"[C02637, C00080, C00005]","[C00006, C00493]",b3281 or b1692,True,R02413,"[1.1.1.25, 1.1.1.282]"


## 2. Get all the protein information

Besides information about the reactions, we need the relation to the proteins in order to establish a protein allocation model. We do this using the gene-protein-reaction association obtained from the BIGG model. The genes is the model can be mapped to the Uniprot accessions, which can be mapped to the KEGG reactions using the Rhea database.

### 2.1 Parse the BIGG gene-protein-reaction associations

In [5]:
#You need to adjust this to find the geneid/locustag for your microbe
#locustag_regex =r'\b([b|s]\d{4})\b'
def extract_b_numbers(text):
    return re.findall(locustag_regex, text)

id_mapper_df['b_number'] = id_mapper_df['GPR'].apply(extract_b_numbers)
id_mapper_df = id_mapper_df.explode('b_number', ignore_index=True)
id_mapper_df

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,ec-code,b_number
0,CYTDK2,"[C00475, C00044]","[C00055, C00035, C00080]",b2066,False,R00517,"[2.7.1.48, 2.7.1.213]",b2066
1,XPPT,"[C00119, C00385]","[C00013, C00655]",b0238,False,R02142,"[2.4.2.8, 2.4.2.22]",b0238
2,HXPRT,"[C00262, C00119]","[C00130, C00013]",b0238 or b0125,False,R01132,2.4.2.8,b0238
3,HXPRT,"[C00262, C00119]","[C00130, C00013]",b0238 or b0125,False,R01132,2.4.2.8,b0125
4,NDPK5,"[C00002, C00361]","[C00008, C00286]",b0474 or b2518,True,R01857,2.7.4.6,b0474
...,...,...,...,...,...,...,...,...
3670,BMOGDS2,"[nan, C00044, C00080]","[nan, C00013]",(b3857 and b3856) or b3857,False,NaN,NaN,b3857
3671,BMOGDS2,"[nan, C00044, C00080]","[nan, C00013]",(b3857 and b3856) or b3857,False,NaN,NaN,b3856
3672,BMOGDS2,"[nan, C00044, C00080]","[nan, C00013]",(b3857 and b3856) or b3857,False,NaN,NaN,b3857
3673,FESD2s,"[nan, C00080, C00533]","[nan, C14819, [C00001, C01328], C00887]",s0001,False,NaN,NaN,s0001


### 2.2 Parse the Uniprot data for merging

In [6]:
# load the uniprot information
uniprot_df = pd.read_excel(UNIPROT_FILE_PATH)
#get the gene id from the gene names
uniprot_df['b_number'] = uniprot_df['Gene Names'].str.extract(locustag_regex)
uniprot_df = uniprot_df.drop('Gene Names', axis=1)
# fill in the missing genes to prevent a mismatch in mapping to the model data
uniprot_df['b_number'] = uniprot_df['b_number'].fillna('missing')
uniprot_df

/home/samiralvdb/Software/anaconda3/envs/PAModelpy/lib/python3.9/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Entry,Entry Name,Length,Mass,Rhea ID,b_number
0,A5A616,MGTS_ECOLI,31,3509,NaN,b4599
1,O32583,THIS_ECOLI,66,7311,NaN,b4407
2,P00350,6PGD_ECOLI,468,51481,RHEA:10116,b2029
3,P00363,FRDA_ECOLI,602,65972,RHEA:40523 RHEA:27834,b4154
4,P00370,DHE4_ECOLI,447,48581,RHEA:11612,b1761
...,...,...,...,...,...,...
4582,Q9S4X4,YUBB_ECOLI,145,16507,NaN,missing
4583,Q9S4X5,YUBA_ECOLI,168,19067,NaN,missing
4584,Q9XB42,YKFH_ECOLI,73,8581,NaN,b4504
4585,Q9Z3A0,YJGW_ECOLI,111,13085,NaN,b4274


### 2.3 Match Uniprot to BIGG data

In [7]:
id_mapper_with_protein = pd.merge(
    id_mapper_df, uniprot_df, on='b_number', how='left'
).rename({'ec-code':'EC', 'b_number': 'locus_tag'}, axis=1)

id_mapper_with_protein

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,locus_tag,Entry,Entry Name,Length,Mass,Rhea ID
0,CYTDK2,"[C00475, C00044]","[C00055, C00035, C00080]",b2066,False,R00517,"[2.7.1.48, 2.7.1.213]",b2066,P0A8F4,URK_ECOLI,213.0,24353.0,RHEA:16825 RHEA:24674
1,XPPT,"[C00119, C00385]","[C00013, C00655]",b0238,False,R02142,"[2.4.2.8, 2.4.2.22]",b0238,P0A9M5,XGPT_ECOLI,152.0,16971.0,RHEA:25424 RHEA:10800 RHEA:17973
2,HXPRT,"[C00262, C00119]","[C00130, C00013]",b0238 or b0125,False,R01132,2.4.2.8,b0238,P0A9M5,XGPT_ECOLI,152.0,16971.0,RHEA:25424 RHEA:10800 RHEA:17973
3,HXPRT,"[C00262, C00119]","[C00130, C00013]",b0238 or b0125,False,R01132,2.4.2.8,b0125,P0A9M2,HPRT_ECOLI,178.0,20115.0,RHEA:17973 RHEA:25424
4,NDPK5,"[C00002, C00361]","[C00008, C00286]",b0474 or b2518,True,R01857,2.7.4.6,b0474,P69441,KAD_ECOLI,214.0,23586.0,RHEA:12973
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3670,BMOGDS2,"[nan, C00044, C00080]","[nan, C00013]",(b3857 and b3856) or b3857,False,NaN,NaN,b3857,P32173,MOBA_ECOLI,194.0,21643.0,RHEA:34243
3671,BMOGDS2,"[nan, C00044, C00080]","[nan, C00013]",(b3857 and b3856) or b3857,False,NaN,NaN,b3856,P32125,MOBB_ECOLI,175.0,19363.0,NaN
3672,BMOGDS2,"[nan, C00044, C00080]","[nan, C00013]",(b3857 and b3856) or b3857,False,NaN,NaN,b3857,P32173,MOBA_ECOLI,194.0,21643.0,RHEA:34243
3673,FESD2s,"[nan, C00080, C00533]","[nan, C14819, [C00001, C01328], C00887]",s0001,False,NaN,NaN,s0001,NaN,NaN,NaN,NaN,NaN


## 3. Match the BiGG model reactiona and enzymes to the kcats from GotEnzymes
Find first all enzymes for a reaction, then for each enzyme determine if the kcats from GotEnzymes are for a forward or reverse reaction

In [8]:
id_mapper_final = id_mapper_with_protein
id_mapper_final = id_mapper_final.rename({'Entry':'enzyme_id'}, axis=1)
id_mapper_final = id_mapper_final.explode('kegg.reaction', ignore_index=True)
id_mapper_final = id_mapper_final.explode('EC', ignore_index=True)

# id_mapper_final = id_mapper_final.drop_duplicates(subset = ['kegg_id'])
id_mapper_final.head()

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,locus_tag,enzyme_id,Entry Name,Length,Mass,Rhea ID
0,CYTDK2,"[C00475, C00044]","[C00055, C00035, C00080]",b2066,False,R00517,2.7.1.48,b2066,P0A8F4,URK_ECOLI,213.0,24353.0,RHEA:16825 RHEA:24674
1,CYTDK2,"[C00475, C00044]","[C00055, C00035, C00080]",b2066,False,R00517,2.7.1.213,b2066,P0A8F4,URK_ECOLI,213.0,24353.0,RHEA:16825 RHEA:24674
2,XPPT,"[C00119, C00385]","[C00013, C00655]",b0238,False,R02142,2.4.2.8,b0238,P0A9M5,XGPT_ECOLI,152.0,16971.0,RHEA:25424 RHEA:10800 RHEA:17973
3,XPPT,"[C00119, C00385]","[C00013, C00655]",b0238,False,R02142,2.4.2.22,b0238,P0A9M5,XGPT_ECOLI,152.0,16971.0,RHEA:25424 RHEA:10800 RHEA:17973
4,HXPRT,"[C00262, C00119]","[C00130, C00013]",b0238 or b0125,False,R01132,2.4.2.8,b0238,P0A9M5,XGPT_ECOLI,152.0,16971.0,RHEA:25424 RHEA:10800 RHEA:17973


In [9]:
#get the json file from GotEnzymes and parse into workable format
#read in the json file obtained from gotEnzymes (for 'eco')
eco_enzymes_json = pd.read_json(GOTENZYMES_FILE_PATH)
#convert to a readible format
eco_enzymes = eco_enzymes_json.enzymes.to_list()
eco_enzymes_df = pd.DataFrame(eco_enzymes)

#ensure there is only one ec number per row
eco_enzymes_df['ec_number'] = eco_enzymes_df['ec_number'].str.split(pat=';')
eco_enzymes_df = eco_enzymes_df.explode('ec_number', ignore_index=True)
# eco_enzymes_df = eco_enzymes_df.dropna(axis=0, subset=['ec_number'])
eco_enzymes_df = eco_enzymes_df.drop(['organism', 'domain'], axis = 1)
eco_enzymes_df.head()

,gene,reaction_id,ec_number,compound,kcat_values
0,b0002,R01773,1.1.1.3,C00263,2.1410
1,b0002,R01773,1.1.1.3,C00441,5.0932
2,b0002,R01775,1.1.1.3,C00263,2.1410
3,b0002,R01775,1.1.1.3,C00441,5.0932
4,b0002,R00480,2.7.2.4,C03082,5.0319


In [10]:
# match BiGG and GotEnzymes based on gene id
eco_enzymes_merged = map_kcat_values_to_reaction_protein_association(id_mapper = id_mapper_final,
                                                                     gotenzymes_df = eco_enzymes_df
                                                                    )
eco_enzymes_merged

Mapped 5925 out of 9313 kcat values to reactions based on kegg gene id
Mapped 4215 out of 9313 kcat values to reactions based on ec number
Final merged dataset has 6127 unique reactions with kcat values.


,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,enzyme_id,Entry Name,Length,Mass,Rhea ID,gene,ec_number,compound,kcat_values
0,CYTDK2,"[C00475, C00044]","[C00055, C00035, C00080]",b2066,False,R00517,2.7.1.48,P0A8F4,URK_ECOLI,213.0,24353.0,RHEA:16825 RHEA:24674,b2066,2.7.1.48,C00055,162.9332
1,CYTDK2,"[C00475, C00044]","[C00055, C00035, C00080]",b2066,False,R00517,2.7.1.48,P0A8F4,URK_ECOLI,213.0,24353.0,RHEA:16825 RHEA:24674,b2066,2.7.1.48,C00475,150.7242
2,CYTDK2,"[C00475, C00044]","[C00055, C00035, C00080]",b2066,False,R00517,2.7.1.213,P0A8F4,URK_ECOLI,213.0,24353.0,RHEA:16825 RHEA:24674,b2066,2.7.1.48,C00055,162.9332
3,CYTDK2,"[C00475, C00044]","[C00055, C00035, C00080]",b2066,False,R00517,2.7.1.213,P0A8F4,URK_ECOLI,213.0,24353.0,RHEA:16825 RHEA:24674,b2066,2.7.1.48,C00475,150.7242
4,XPPT,"[C00119, C00385]","[C00013, C00655]",b0238,False,R02142,2.4.2.8,P0A9M5,XGPT_ECOLI,152.0,16971.0,RHEA:25424 RHEA:10800 RHEA:17973,b0238,2.4.2.8,C00655,0.0056
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9242,BMOGDS2,"[nan, C00044, C00080]","[nan, C00013]",(b3857 and b3856) or b3857,False,NaN,NaN,P32173,MOBA_ECOLI,194.0,21643.0,RHEA:34243,b3857,NaN,NaN,NaN
9243,BMOGDS2,"[nan, C00044, C00080]","[nan, C00013]",(b3857 and b3856) or b3857,False,NaN,NaN,P32125,MOBB_ECOLI,175.0,19363.0,NaN,b3856,NaN,NaN,NaN
9244,BMOGDS2,"[nan, C00044, C00080]","[nan, C00013]",(b3857 and b3856) or b3857,False,NaN,NaN,P32173,MOBA_ECOLI,194.0,21643.0,RHEA:34243,b3857,NaN,NaN,NaN
9245,FESD2s,"[nan, C00080, C00533]","[nan, C14819, [C00001, C01328], C00887]",s0001,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,s0001,NaN,NaN,NaN


### 6. Extract directionalities and gapfill
Reactions which are associated with an enzyme, but not with a kcat from GotEnzymes will be assignes
a default kcat and if required a default molmass.

In [11]:
# Set default values
eco_enzymes_parsed = assign_directionalities_for_kcat_relations(eco_enzymes_merged.copy())
eco_enzymes_parsed = assign_defaults_for_proteins_without_mapping(eco_enzymes_parsed)
eco_enzymes_parsed

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,enzyme_id,Entry Name,Length,molMass,Rhea ID,gene,kcat_values,direction
0,CYTDK2,"[C00475, C00044]","[C00055, C00035, C00080]",b2066,False,R00517,2.7.1.48,P0A8F4,URK_ECOLI,213.0,24353.0000,RHEA:16825 RHEA:24674,b2066,162.9332,b
1,CYTDK2,"[C00475, C00044]","[C00055, C00035, C00080]",b2066,False,R00517,2.7.1.48,P0A8F4,URK_ECOLI,213.0,24353.0000,RHEA:16825 RHEA:24674,b2066,150.7242,f
2,CYTDK2,"[C00475, C00044]","[C00055, C00035, C00080]",b2066,False,R00517,2.7.1.213,P0A8F4,URK_ECOLI,213.0,24353.0000,RHEA:16825 RHEA:24674,b2066,162.9332,b
3,CYTDK2,"[C00475, C00044]","[C00055, C00035, C00080]",b2066,False,R00517,2.7.1.213,P0A8F4,URK_ECOLI,213.0,24353.0000,RHEA:16825 RHEA:24674,b2066,150.7242,f
4,XPPT,"[C00119, C00385]","[C00013, C00655]",b0238,False,R02142,2.4.2.8,P0A9M5,XGPT_ECOLI,152.0,16971.0000,RHEA:25424 RHEA:10800 RHEA:17973,b0238,0.0056,b
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9572,URIt2pp_copy2,"[C00080, C00299]","[C00080, C00299]",b2406,True,NaN,NaN,P45562,XAPB_ECOLI,418.0,46140.0000,RHEA:28939,b2406,13.7000,b
9573,XANtpp,[C00385],[C00385],Gene_XANtpp,True,NaN,NaN,Enzyme_XANtpp,NaN,325.0,39959.4825,NaN,Gene_XANtpp,13.7000,b
9574,XTSNt2rpp,"[C00080, C01762]","[C00080, C01762]",b2406,True,NaN,NaN,P45562,XAPB_ECOLI,418.0,46140.0000,RHEA:28939,b2406,13.7000,b
9575,XYLI2,[C00031],"[[C01496, C05003, C00095, C10906]]",b3565,True,R00307,5.3.1.5,P00944,XYLA_ECOLI,440.0,49742.0000,RHEA:22816,b3565,22.2245,b


In [12]:
protein2gene, gene2protein = get_protein_gene_mapping(eco_enzymes_parsed, model)


# Ensure the enzyme complexes are merged on one row
eco_enzymes_mapped = merge_enzyme_complexes(eco_enzymes_parsed, gene2protein)
eco_enzymes_mapped = eco_enzymes_mapped.drop_duplicates(['rxn_id', 'enzyme_id', 'direction'])

# Save the adjusted dataframe or continue processing
eco_enzymes_mapped = eco_enzymes_mapped.drop(['Entry Name', 'Rhea ID', 'reversible'], axis=1)
eco_enzymes_mapped

,rxn_id,Reactants,Products,GPR,kegg.reaction,EC,enzyme_id,Length,molMass,gene,kcat_values,direction
8284,12DGR120tipp,[nan],[nan],Gene_12DGR120tipp,NaN,NaN,Enzyme_12DGR120tipp,325.0,39959.4825,[Gene_12DGR120tipp],13.7,f
8248,12DGR140tipp,[nan],[nan],Gene_12DGR140tipp,NaN,NaN,Enzyme_12DGR140tipp,325.0,39959.4825,[Gene_12DGR140tipp],13.7,f
7858,12DGR141tipp,[nan],[nan],Gene_12DGR141tipp,NaN,NaN,Enzyme_12DGR141tipp,325.0,39959.4825,[Gene_12DGR141tipp],13.7,f
8250,12DGR160tipp,[nan],[nan],Gene_12DGR160tipp,NaN,NaN,Enzyme_12DGR160tipp,325.0,39959.4825,[Gene_12DGR160tipp],13.7,f
8521,12DGR161tipp,[nan],[nan],Gene_12DGR161tipp,NaN,NaN,Enzyme_12DGR161tipp,325.0,39959.4825,[Gene_12DGR161tipp],13.7,f
...,...,...,...,...,...,...,...,...,...,...,...,...
7362,ZN2abcpp,"[C00002, [C00001, C01328], C00038]","[C00008, C00080, C00009, C00038]",b3469,NaN,3.6.3.5,P37617,732.0,76840.0000,[b3469],13.7,f
7521,ZN2t3pp,"[C00080, C00038]","[C00080, C00038]",b3915 or b0752,NaN,NaN,P69380,300.0,32927.0000,[b3915],13.7,f
7522,ZN2t3pp,"[C00080, C00038]","[C00080, C00038]",b3915 or b0752,NaN,NaN,P75757,313.0,34678.0000,[b0752],13.7,f
8548,ZN2tpp,[C00038],[C00038],b3040,NaN,NaN,P0A8H3,257.0,26485.0000,[b3040],13.7,f


In [13]:
#drop duplicate entries. If there are duplicates, make sure the mean kcat value is used to parametrize
eco_enzymes_final = eco_enzymes_mapped.groupby(
    ['rxn_id', 'enzyme_id', 'direction'], 
    as_index=False
).agg({
    'kcat_values': 'mean', 
    **{col: 'first' for col in eco_enzymes_mapped.columns if col not in ['rxn_id', 'enzyme_id', 'direction', 'kcat']
      }
})
# eco_enzymes_final.to_excel(AE_OUTPUT_FILE_PATH)

### 7. Save the dataframe to the proper format for building PAMs

In [14]:
# Get the information about the enzyme sectors
other_sectors = pd.read_excel(PAM_DATA_FILE_PATH,
                                 sheet_name = None)
del other_sectors['ActiveEnzymes']

In [15]:
# Save it to a new excel file
with pd.ExcelWriter(OUTPUT_FILE_PATH) as writer:
    eco_enzymes_final.to_excel(writer, sheet_name='ActiveEnzymes', index = False)
    for sheet, df in other_sectors.items():
        if sheet == 'ExcessEnzymes': sheet = 'UnusedEnzyme'
        df.to_excel(writer, sheet_name=sheet, index = False)